In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
import os, time, json, glob, shutil

fablib = fablib_manager()
fablib.show_config()

fablib.list_sites(
    fields=["name", "state", "cores_available", "ram_available", "disk_available",
            "gpu_rtx6000_available", "gpu_tesla_t4_available",
            "gpu_a30_available", "gpu_a40_available"]
)

In [ ]:
def _df(obj):
    return obj.data if hasattr(obj, "data") else obj

def host_gpu_table(min_free=1):
    df = _df(fablib.list_hosts(output="pandas", quiet=True))
    cols = list(df.columns)
    def find(*subs):
        for c in cols:
            if all(s in c.lower() for s in subs): return c
        return None
    name_col = find("name") or find("host") or cols[0]
    avail = {
        "A40": find("a40", "avail") or find("a40"),
        "A30": find("a30", "avail") or find("a30"),
        "RTX6000": find("rtx6000", "avail") or find("rtx6000"),
        "TESLA": find("teslat4", "avail") or find("tesla"),
        "Cores": find("core", "avail") or find("cpu", "avail"),
        "RAM": find("ram", "avail"),
        "Disk": find("disk", "avail"),
    }
    def num(r, col):
        try: return int(float(r[col]))
        except Exception: return 0
    rows = []
    for _, r in df.iterrows():
        e = {"Host": r[name_col]}
        for k, c in avail.items(): e[k] = num(r, c) if c else 0
        rows.append(e)
    rows = [e for e in rows
            if max(e["A40"], e["A30"], e["RTX6000"], e["TESLA"]) >= min_free
            and e["Cores"] >= 16]
    rows.sort(key=lambda e: (e["A40"], e["A30"], e["RTX6000"], e["TESLA"]), reverse=True)
    return rows

hosts = host_gpu_table()
for e in hosts:
    print(f"{str(e['Host'])[:32]:32s} A40 {e['A40']} A30 {e['A30']} RTX6k {e['RTX6000']} "
          f"T4 {e['TESLA']} cores {e['Cores']} ram {e['RAM']}")

def pick_host(gpu):
    for e in hosts:
        if e[gpu] >= 1:
            site = str(e["Host"]).split("-")[0].split(".")[0].upper()
            return {"host": e["Host"], "site": site, "free": e[gpu], "cores": e["Cores"]}
    return None

print("\n--- best host per GPU type ---")
for g in ["A40", "A30", "RTX6000", "TESLA"]:
    print(f"{g:8s} ->", pick_host(g) or "none free")

In [ ]:
# SLICE_NAME = "ICC-attack-falcon"
# SITE       = "CERN"
# HOST       = "cern-w4.fabric-testbed.net"
# GPU_MODEL  = "GPU_A30"

#Run-2
SLICE_NAME = "ICC-attack-falcon-run2"
SITE       = "CERN"
HOST       = "cern-w4.fabric-testbed.net"
GPU_MODEL  = "GPU_A30"

slice_obj = fablib.new_slice(name=SLICE_NAME)
node = slice_obj.add_node(name="gpu-node", site=SITE, host=HOST,
                          cores=16, ram=64, disk=200, image="default_ubuntu_22")
node.add_component(model=GPU_MODEL, name="gpu1")
slice_obj.submit()
print(slice_obj)

In [ ]:
# #copying form llama one
# LOCAL     = os.getcwd()                                      # Falcon notebook folder
# LLAMA_DIR = os.path.expanduser("~/work/ICC - TCP|ACK LLAMA")
# LOCAL_OUT = os.path.join(LOCAL, "out_falcon")
# os.makedirs(LOCAL_OUT, exist_ok=True)

# shutil.copy(os.path.join(LLAMA_DIR, "gen_attack.py"), os.path.join(LOCAL, "gen_attack.py"))

# src = open(os.path.join(LOCAL, "gen_attack.py")).read()
# print("files:", sorted(os.listdir(LOCAL)))
# print("JSON driver:", '"format"' in src)

#Run-2

LOCAL     = os.getcwd()                                      # Falcon notebook folder
LOCAL_OUT = os.path.join(LOCAL, "out_falcon_run2")
os.makedirs(LOCAL_OUT, exist_ok=True)

src = open(os.path.join(LOCAL, "gen_attack.py")).read()
print("files:", sorted(os.listdir(LOCAL)))
print("JSON driver:", '"format"' in src)

In [ ]:
#Slice Extend - run2

from datetime import datetime, timezone, timedelta

slice_obj = fablib.get_slice(SLICE_NAME)
end = (datetime.now(timezone.utc) + timedelta(days=5)).strftime("%Y-%m-%d %H:%M:%S %z")
slice_obj.renew(end)
print("new lease end:", fablib.get_slice(SLICE_NAME).get_lease_end())

In [ ]:
slice_obj = fablib.get_slice(SLICE_NAME)
node = slice_obj.get_node("gpu-node")
print("mgmt ip:", node.get_management_ip())

REMOTE = "/home/ubuntu/llmdal"
MODEL  = "falcon3:10b"
out, err = node.execute("lspci | grep -i nvidia")

In [ ]:
node.execute("sudo apt-get update -q && "
             "sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -q nvidia-driver-535",
             quiet=True)
node.execute("sudo reboot", quiet=True)
print("rebooting, waiting 150s...")
time.sleep(150)
out, err = node.execute("nvidia-smi --query-gpu=name,memory.total,driver_version "
                        "--format=csv,noheader")

In [ ]:
setup = [
    #"which ollama >/dev/null 2>&1 || (curl -fsSL https://ollama.com/install.sh | sh)",
    #run-2
    "which ollama >/dev/null 2>&1 || (curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.34.4 sh)",
    "sudo mkdir -p /etc/systemd/system/ollama.service.d",
    "printf '[Service]\\nEnvironment=\"OLLAMA_MAX_LOADED_MODELS=1\"\\n"
    "Environment=\"OLLAMA_NUM_PARALLEL=1\"\\n"
    "Environment=\"OLLAMA_KEEP_ALIVE=-1\"\\n' | "
    "sudo tee /etc/systemd/system/ollama.service.d/limits.conf",
    "sudo systemctl daemon-reload && sudo systemctl enable --now ollama && "
    "sudo systemctl restart ollama",
    "sleep 5; curl -s http://127.0.0.1:11434/api/version",
]
for c in setup:
    out, err = node.execute(c, quiet=True)
    if out.strip(): print(out.strip()[-200:])

out, err = node.execute(f"ollama pull {MODEL} 2>&1 | tail -2")
out, err = node.execute("ollama list")

In [ ]:
node.execute(f"mkdir -p {REMOTE}/prompts {REMOTE}/out", quiet=True)
node.upload_file(os.path.join(LOCAL, "gen_attack.py"),         f"{REMOTE}/gen_attack.py")
node.upload_file(os.path.join(LOCAL, "prompt_attack.txt"),     f"{REMOTE}/prompts/prompt_attack.txt")
node.upload_file(os.path.join(LOCAL, "generation_order.json"), f"{REMOTE}/prompts/generation_order.json")
out, err = node.execute(f"ls -la {REMOTE} {REMOTE}/prompts")

In [ ]:
node.execute(f"rm -f {REMOTE}/out/sanity.*", quiet=True)
out, err = node.execute(
    f"cd {REMOTE} && MODEL={MODEL} K=1 MAX_CALLS=1 NUM_PREDICT=3000 "
    f"TAG=sanity python3 gen_attack.py")
out, err = node.execute(f"head -c 3000 {REMOTE}/out/sanity.raw.txt")

In [ ]:
# node.execute(f"rm -f {REMOTE}/out/sanity.* {REMOTE}/out/smoke_* {REMOTE}/smoke.log", quiet=True)

# SMOKE = (f"cd {REMOTE} && "
#          f"MODEL={MODEL} K=1 TARGET_ROWS=5 MAX_CALLS=10 NUM_PREDICT=3000 "
#          f"TAG=smoke_falcon_k1 python3 -u gen_attack.py > smoke.log 2>&1")
# node.execute(f"setsid nohup bash -c '{SMOKE}' > /dev/null 2>&1 < /dev/null &", quiet=True)
# time.sleep(10)
# out, err = node.execute(f"tail -3 {REMOTE}/smoke.log")

#Run-2 for K=1, 5 and 10

import json, numpy as np, pandas as pd

RULES = json.load(open(os.path.join(LOCAL, "rules.json")))
EMPTY = {"", "na", "nan", "none", "null"}
ID = ["ip_a", "ip_b", "port_a", "port_b", "flow_key", "timeframe_id"]

def num(s):
    s = s.where(~s.astype(str).str.strip().str.lower().isin(EMPTY))
    return pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)

def same(a, b):   # equal within 0.1%, or both empty
    return (np.isnan(a) & np.isnan(b)) | np.isclose(a, b, rtol=1e-3, atol=1e-6)

def check(tag):
    for f in [f"{tag}.csv", f"{tag}.log.jsonl"]:
        node.download_file(os.path.join(LOCAL_OUT, f), f"{REMOTE}/out/{f}")
    L = pd.DataFrame([json.loads(l) for l in open(f"{LOCAL_OUT}/{tag}.log.jsonl") if l.strip()])
    D = pd.read_csv(f"{LOCAL_OUT}/{tag}.csv", dtype=str, keep_default_na=False)
    env = {c: num(D[c]) for c in D.columns if c not in ("flow_key", "ip_a", "ip_b", "is_attack")}

    # 1) constants: 16 pinned values + is_attack = True
    const = np.ones(len(D), bool)
    for col, v in RULES["pinned"].items():
        const &= (D[col] == v).to_numpy() if isinstance(v, str) else same(env[col], np.full(len(D), float(v)))
    const &= (D["is_attack"].str.lower() == "true").to_numpy()

    # 2) derived: 42 formulas + flow_key
    der = np.ones(len(D), bool)
    with np.errstate(divide="ignore", invalid="ignore"):
        for col, f in RULES["derived"].items():
            exp = np.array(np.broadcast_to(eval(f, {}, env), len(D)), dtype=float)
            exp[~np.isfinite(exp)] = np.nan
            der &= same(env[col], exp)
    der &= (D["flow_key"] == D[RULES["flow_key_rule"].split("|")].agg("|".join, axis=1)).to_numpy()

    # 3) duplicates: exact (dropped by driver) + near (same row apart from IP/port/key/timeframe)
    K, calls, valid = int(L["k"].iloc[0]), len(L), int(L["valid"].sum())
    near = D.drop(columns=ID).duplicated().to_numpy()
    dups = int(L["dup"].sum()) + int(near.sum())

    total = L["sec"].sum()
    good = int((const & der & ~near).sum())   # passes everything and isn't a near-copy
    r = {"tag": tag, "K": K, "calls": calls,
         "complete %": 100 * valid / max(K * calls, 1),      # 113 columns returned
         "constants %": 100 * const.mean() if len(D) else 0,
         "derived %": 100 * der.mean() if len(D) else 0,
         "duplicate %": 100 * dups / max(valid, 1),
         "total time (min)": total / 60,
         "good rows": good,
         "sec per good row": total / max(good, 1)}
    r["hrs for 3300"] = r["sec per good row"] * 3300 / 3600
    r["PASS"] = (r["complete %"] >= 90 and r["constants %"] >= 95
                 and r["derived %"] >= 95 and r["duplicate %"] < 5)
    return r

In [ ]:
#run-2 sanity check for smoke test
SHORT = "falcon"

SWEEP = (f"cd {REMOTE} && for k in 1 5 10; do "
         f"NP=$((k*2500+500)); [ $NP -gt 20000 ] && NP=20000; "
         f"MODEL={MODEL} K=$k TARGET_ROWS=$((k*10)) MAX_CALLS=10 "
         f"NUM_PREDICT=$NP TAG=smoke2_{SHORT}_k$k python3 -u gen_attack.py; "
         f"done > smoke2.log 2>&1")
node.execute(f"setsid nohup bash -c '{SWEEP}' > /dev/null 2>&1 < /dev/null &", quiet=True)
time.sleep(30)
out, err = node.execute(f"tail -3 {REMOTE}/smoke2.log")

In [ ]:
#run-2
import os, json, numpy as np, pandas as pd

SHORT     = "falcon"
REMOTE    = "/home/ubuntu/llmdal"
LOCAL     = os.getcwd()
LOCAL_OUT = os.path.join(LOCAL, f"out_{SHORT}_run2")
os.makedirs(LOCAL_OUT, exist_ok=True)

RULES = json.load(open(os.path.join(LOCAL, "rules.json")))
EMPTY = {"", "na", "nan", "none", "null"}
ID = ["ip_a", "ip_b", "port_a", "port_b", "flow_key", "timeframe_id"]

def num(s):
    s = s.where(~s.astype(str).str.strip().str.lower().isin(EMPTY))
    return pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)

def same(a, b):
    return (np.isnan(a) & np.isnan(b)) | np.isclose(a, b, rtol=1e-3, atol=1e-6)

def check(tag):
    for f in [f"{tag}.csv", f"{tag}.log.jsonl"]:
        node.download_file(os.path.join(LOCAL_OUT, f), f"{REMOTE}/out/{f}")
    L = pd.DataFrame([json.loads(l) for l in open(f"{LOCAL_OUT}/{tag}.log.jsonl") if l.strip()])
    D = pd.read_csv(f"{LOCAL_OUT}/{tag}.csv", dtype=str, keep_default_na=False)
    env = {c: num(D[c]) for c in D.columns if c not in ("flow_key", "ip_a", "ip_b", "is_attack")}

    # 1) constants: 16 pinned values + is_attack = True
    const = np.ones(len(D), bool)
    for col, v in RULES["pinned"].items():
        const &= (D[col] == v).to_numpy() if isinstance(v, str) else same(env[col], np.full(len(D), float(v)))
    const &= (D["is_attack"].str.lower() == "true").to_numpy()

    # 2) derived: each of the 42 formulas + flow_key, scored separately
    hits = {}
    with np.errstate(divide="ignore", invalid="ignore"):
        for col, f in RULES["derived"].items():
            exp = np.array(np.broadcast_to(eval(f, {}, env), len(D)), dtype=float)
            exp[~np.isfinite(exp)] = np.nan
            hits[col] = same(env[col], exp)
    hits["flow_key"] = (D["flow_key"] == D[RULES["flow_key_rule"].split("|")].agg("|".join, axis=1)).to_numpy()
    H = pd.DataFrame(hits)

    # 3) duplicates: exact (dropped by driver) + near (same row apart from IP/port/key/timeframe)
    K, calls, valid = int(L["k"].iloc[0]), len(L), int(L["valid"].sum())
    near = D.drop(columns=ID).duplicated().to_numpy()
    dups = int(L["dup"].sum()) + int(near.sum())

    total = L["sec"].sum()
    r = {"tag": tag, "K": K, "calls": calls, "rows": len(D),
         "complete %": 100 * valid / max(K * calls, 1),
         "constants %": 100 * const.mean() if len(D) else 0,
         "formulas correct %": 100 * H.values.mean() if len(D) else 0,
         "duplicate %": 100 * dups / max(valid, 1),
         "total time (min)": total / 60,
         "sec per row": total / max(len(D), 1)}
    r["hrs for 3300"] = r["sec per row"] * 3300 / 3600
    r["PASS"] = (r["complete %"] >= 90 and r["constants %"] >= 95 and r["duplicate %"] < 5)
    return r

res = pd.DataFrame([check(f"smoke2_{SHORT}_k{k}") for k in [1, 5, 10]]).round(2)
print(res.set_index("tag").T.to_string())

In [ ]:
path = os.path.join(LOCAL_OUT, f"smoke_test_results_{SHORT}.csv")
res.to_csv(path, index=False)
print("saved:", path)
print(pd.read_csv(path).set_index("tag").T.to_string())

In [ ]:
#run-1
import numpy as np, pandas as pd

out, err = node.execute(f"ls {REMOTE}/out", quiet=True)
for f in out.split():
    if f.startswith("smoke_") or f.startswith("sanity"):
        node.download_file(os.path.join(LOCAL_OUT, f), f"{REMOTE}/out/{f}")

def num(s): return pd.to_numeric(s.replace("NA", np.nan), errors="coerce")

for log in sorted(glob.glob(f"{LOCAL_OUT}/smoke_*.log.jsonl")):
    tag = os.path.basename(log).replace(".log.jsonl", "")
    L = pd.DataFrame([json.loads(l) for l in open(log)])
    D = pd.read_csv(f"{LOCAL_OUT}/{tag}.csv", dtype=str, keep_default_na=False)
    pf, pb, pt = num(D.pkts_fwd), num(D.pkts_bwd), num(D.pkts_total)
    bt, dur = num(D.bytes_total), num(D.duration)
    fk = (D.timeframe_id + "|" + D.proto + "|" + D.ip_a + "|" + D.ip_b +
          "|" + D.port_a + "|" + D.port_b)
    print(f"{tag}: {len(D)} rows, {len(L)} calls, {L.sec.mean():.0f}s/call")
    print(f"  pkts_total ok    {100*(pt == pf + pb).mean():.0f}%")
    print(f"  mean_pkt_size ok {100*np.isclose(num(D.mean_pkt_size), bt/pt, rtol=1e-3).mean():.0f}%")
    print(f"  pps_fwd ok       {100*np.isclose(num(D.pps_fwd), pf/dur, rtol=1e-3).mean():.0f}%")
    print(f"  flow_key ok      {100*(D.flow_key == fk).mean():.0f}%")
    print(f"  is_attack True   {100*(D.is_attack.str.lower() == 'true').mean():.0f}%")
    print(f"  mean_pkt_size > 1500 (over MTU): {(num(D.mean_pkt_size) > 1500).sum()}/{len(D)}")

In [ ]:
# from datetime import datetime, timezone, timedelta

# print("before:", slice_obj.get_lease_end())
# end = (datetime.now(timezone.utc) + timedelta(days=7)).strftime("%Y-%m-%d %H:%M:%S %z")
# slice_obj.renew(end)

# slice_obj = fablib.get_slice(SLICE_NAME)
# node = slice_obj.get_node("gpu-node")
# print("after: ", slice_obj.get_lease_end())
#Run-1

In [ ]:
FULL = (f"cd {REMOTE} && "
        f"MODEL={MODEL} K=1 TARGET_ROWS=3300 NUM_PREDICT=3000 "
        f"TAG=falcon_full python3 -u gen_attack.py > full.log 2>&1")
node.execute(f"setsid nohup bash -c '{FULL}' > /dev/null 2>&1 < /dev/null &", quiet=True)
time.sleep(60)
out, err = node.execute(f"tail -3 {REMOTE}/full.log")

In [ ]:
import os, pandas as pd

LOCAL_OUT = os.path.join(os.getcwd(), "out_falcon")
os.makedirs(LOCAL_OUT, exist_ok=True)

out, err = node.execute(f"tail -2 {REMOTE}/full.log")
for f in ["falcon_full.csv", "falcon_full.log.jsonl", "falcon_full.raw.txt"]:
    node.download_file(os.path.join(LOCAL_OUT, f), f"{REMOTE}/out/{f}")

D = pd.read_csv(os.path.join(LOCAL_OUT, "falcon_full.csv"))
print(sorted(os.listdir(LOCAL_OUT)))
print("rows:", len(D), "| cols:", D.shape[1], "| dup rows:", D.duplicated().sum())

In [ ]:
#### Final Run-2 #############
TAG = "falcon_run2"

FULL = (f"cd {REMOTE} && "
        f"MODEL={MODEL} K=1 TARGET_ROWS=3300 NUM_PREDICT=3000 SEED_BASE=100000 "
        f"TAG={TAG} python3 -u gen_attack.py > {TAG}.log 2>&1")
node.execute(f"setsid nohup bash -c '{FULL}' > /dev/null 2>&1 < /dev/null &", quiet=True)
time.sleep(60)
out, err = node.execute(f"tail -3 {REMOTE}/{TAG}.log; echo; cat {REMOTE}/out/{TAG}.settings.json")
